# CMPS 355 — Phase 2 Overlap Null (C2: Pairwise + 4-Way + 95% MC Intervals) & Fixed-K (C3) v3

**GATED V3 STAGING NOTEBOOK — DO NOT SET `ALLOW_FULL_RUN = True` UNTIL CHATGPT INDEPENDENT V3 SIGN-OFF PASSES AND C1 V3 IS `CERTIFIED_COMPLETE`.**

- Expected `phase2_overlap_and_size_v3.py` SHA-256: `126d64b5193ff4855304c3ad008c3078c88ea7b9e36673b5b49feb20fcab4be8`
- Expected `overlap_manifest_config_v3.json` SHA-256: `d2ca01eb7baa056d3d20ccbf23b7eadd819c0081ca97f2b5b7feacc8ef3e9374`
- Expected `phase2_baselines_v3.py` SHA-256: `14387756ef1c85f0a2bf8c1cf0a108bc2575cec645365ae180e48b9c7ce6a1e1`
- Immutable canonical ZIP SHA-256: `8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, os, subprocess, sys
from pathlib import Path

BASE_EXP = Path('/content/drive/MyDrive/CMPS355_Research/peer_review_revision_2026/03_followup_experiments')
C1_DIR = BASE_EXP / 'phase2_baselines_20261122_v3'
C2_DIR = BASE_EXP / 'phase2_overlap_and_size_20261122_v3'
SCRIPT_PATH = C2_DIR / 'phase2_overlap_and_size_v3.py'
CONFIG_PATH = C2_DIR / 'overlap_manifest_config_v3.json'
C1_SCRIPT_PATH = C1_DIR / 'phase2_baselines_v3.py'
DATA_ZIP = Path('/content/drive/MyDrive/CMPS355_Research/data/validated/ibd_dataset_sample_aligned.zip')
OUT_DIR = C2_DIR / 'outputs'

EXPECTED_C2_SHA = '126d64b5193ff4855304c3ad008c3078c88ea7b9e36673b5b49feb20fcab4be8'
EXPECTED_CFG_SHA = 'd2ca01eb7baa056d3d20ccbf23b7eadd819c0081ca97f2b5b7feacc8ef3e9374'
EXPECTED_C1_SHA = '14387756ef1c85f0a2bf8c1cf0a108bc2575cec645365ae180e48b9c7ce6a1e1'
EXPECTED_DATA_SHA = '8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

for label, path, exp in [('C2/C3 v3 script', SCRIPT_PATH, EXPECTED_C2_SHA), ('Pinned manifest v3', CONFIG_PATH, EXPECTED_CFG_SHA), ('C1 v3 helper', C1_SCRIPT_PATH, EXPECTED_C1_SHA), ('Canonical ZIP', DATA_ZIP, EXPECTED_DATA_SHA)]:
    assert path.is_file(), f'Missing {label}: {path}'
    got = sha256_file(path)
    assert got == exp, f'SHA mismatch for {label}: {got}'
    print(f'OK {label}: {got}')


In [ ]:
# Cell 2: FULL PHASE 2 C2 + C3 V3 EXECUTION — STRICTLY GATED
ALLOW_FULL_RUN = True
if not ALLOW_FULL_RUN:
    raise RuntimeError('BLOCKED: ALLOW_FULL_RUN=False. Requires explicit ChatGPT authorization.')
assert sha256_file(SCRIPT_PATH) == EXPECTED_C2_SHA and sha256_file(CONFIG_PATH) == EXPECTED_CFG_SHA
assert sha256_file(C1_SCRIPT_PATH) == EXPECTED_C1_SHA and sha256_file(DATA_ZIP) == EXPECTED_DATA_SHA
env = os.environ.copy()
env['PYTHONPATH'] = str(C1_DIR) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
import time
t0 = time.time()
subprocess.run([sys.executable, str(SCRIPT_PATH), '--config', str(CONFIG_PATH), '--data', str(DATA_ZIP), '--out', str(OUT_DIR), '--n-mc', '10000', '--n-boot', '2000', '--seed', '20261122'], check=True, env=env)
print(f'Full C2/C3 v3 finished in {time.time() - t0:.2f}s')


In [ ]:
# Cell 3: Fail-closed --finalize-only verification
RUN_FINALIZE_ONLY = False
if not RUN_FINALIZE_ONLY:
    raise RuntimeError('BLOCKED: RUN_FINALIZE_ONLY=False.')
assert sha256_file(SCRIPT_PATH) == EXPECTED_C2_SHA and sha256_file(CONFIG_PATH) == EXPECTED_CFG_SHA
subprocess.run([sys.executable, str(SCRIPT_PATH), '--config', str(CONFIG_PATH), '--out', str(OUT_DIR), '--finalize-only'], check=True)
